# Chapter 6 — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Chapter 6 - Fine-tuning for classification** of *'Build a Large Language Model (From Scratch)'* book by Sebastian Raschka.

> **Attribution:** Portions of the code in this notebook follow and adapt the Apache-2.0-licensed implementation accompanying Sebastian Raschka's *Build a Large Language Model (From Scratch)*.  
> Original source code: https://github.com/rasbt/LLMs-from-scratch  
> Additional annotations, experiments, explanations, and study notes were created as part of my own learning and implementation process.

### 0. Chapter Objective

Fine-tune a pretrained GPT model for binary text classification by preparing labeled data, replacing the language-model output head with a classification head, and training the model to predict `spam` or `not spam`.

### 1. Dataset Preparation: create_balanced_dataset() and random_split() functions

**Purpose:**

- `create_balanced_dataset()` undersamples the majority `ham` class to match the
  number of `spam` examples.
- `random_split()` shuffles and divides the balanced data into training,
  validation, and test sets.

In [ ]:
import pandas as pd
# Downsampling the majority class 'ham' class (because 'spam' class has way less samples)
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)

    balanced_df = pd.concat( [ham_subset, df[df["Label"] == "spam"]] ) # stacking them vertically
    return balanced_df

# balanced_df = create_balanced_dataset(df)
# print(balanced_df["Label"].value_counts())

def random_split(df, train_frac, validation_frac):
    df = df.sample(frac = 1, random_state=123).reset_index(drop=True) # shuffling the df
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

# train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

### 2. SpamDataset class

`SpamDataset` converts each text message into token IDs, pads or truncates sequences to a common length, and returns the encoded text together with its class label.

```text
SMS text
→ tokenize
→ token IDs
→ pad / truncate
→ input tensor + class label
```

In [ ]:
import torch
from torch.utils.data import Dataset

# This ensures each input tensor is of the same size, which is necessary to create the batches in the training data loader we implement next
class SpamDataset(Dataset): # pads the shortest texts to have the same length as the longest (or truncates the texts to a preset max_length)
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256): # Note that max_length could be smaller than self._longest_encoded_length()
        self.data = pd.read_csv(csv_file) # this is a pandas DataFrame with columns: "Label" and "Text"
        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]
        if max_length is None:
            self.max_length = self._longest_encoded_length() # the absolute max length of the encoded texts
        else:
            self.max_length = max_length # which is a set number which could be less than the real maximum length
            self.encoded_texts = [encoded_text[:self.max_length] for encoded_text in self.encoded_texts] # i.e possibly truncating encoded_texts to the preset max_length
        self.encoded_texts = [encoded_text + [pad_token_id]*(self.max_length-len(encoded_text)) for encoded_text in self.encoded_texts] # Padding 

    def __getitem__(self, index):
        encoded = self.encoded_texts[index] # gets the list of tokenIDs corresponding to the Text at row index
        label = self.data.iloc[index]["Label"] # is a string: either "ham" or "spam"
        return (torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long)) # Note: dtype=torch.long expects integer values

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if max_length < encoded_length:
                max_length = encoded_length
        return max_length

# train_dataset = SpamDataset(
#     csv_file="../data/train.csv",
#     tokenizer=tokenizer,
#     max_length=None
# )

# print(train_dataset.max_length)

### 3. Adapting GPT for Classification

A pretrained GPT model normally produces vocabulary-sized logits for every token position:

```text
[b, n, vocab_size]
```

For binary classification, the language-model output head is replaced with a
two-class linear layer:

```python
model.out_head = torch.nn.Linear(
    in_features=BASE_CONFIG["emb_dim"],
    out_features=2
)
```

The classifier uses the logits from the last token position:
```text
[b, n, 2]
    ↓
last token
    ↓
[b, 2]
```

With causal attention, the last token can incorporate information from all preceding tokens, making it a useful representation of the complete sequence.



### 4. Classification Loss and Accuracy

#### 4.1. calc_accuracy_loader() function

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None): 
    model.eval() # disabling Dropout layers! (We will be doing inference below)
    correct_predictions, num_examples = 0, 0

    if num_batches is None: # i.e. if num_batches is NOT specified
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            with torch.no_grad():
                logits = model(input_batch)[:, -1, :] # doing inference! we only care about the last token!
            predicted_labels = torch.argmax(logits, dim=-1) # getting the index of the highest number - shape: (batch_size, seq_len)
            num_examples += predicted_labels.shape[0] # i.e. the batch_size 
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break

    return correct_predictions / num_examples # accuracy metric

#### 4.2 calc_loss_batch() and calc_loss_loader() functions

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

# with torch.no_grad():
#     train_loss = calc_loss_loader(
#         train_loader, model, device, num_batches=5
#     )
#     val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
#     test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)
# print(f"Training loss: {train_loss:.3f}")
# print(f"Validation loss: {val_loss:.3f}")
# print(f"Test loss: {test_loss:.3f}")

**Classification loss**

```text
input IDs
→ GPT classifier
→ logits [b, n, 2]
→ last-token logits [b, 2]
→ cross-entropy with class labels [b]
```
**Accuracy**  
```text
last-token logits
→ argmax
→ predicted class
→ compare with target label
```

### 5. Training the Classifier

The loop can be summarized as:
```
batch
→ forward pass
→ last-token classification logits
→ cross-entropy loss
→ backward()
→ optimizer.step()
→ periodically evaluate loss and accuracy
```

### 6. Classifying New Text
A short mental model:
```
new text
→ tokenize
→ pad / truncate
→ model
→ last-token logits
→ argmax
→ spam / not spam
```

### 7. Chapter 6 Flow

```text
Raw SMS dataset
        ↓
Balance classes
        ↓
Train / Validation / Test split
        ↓
SpamDataset + DataLoader
        ↓
Pretrained GPT-2
        ↓
Replace vocabulary head with 2-class head
        ↓
Fine-tune selected layers
        ↓
Last-token logits [b, 2]
        ↓
Cross-entropy loss
        ↓
spam / not spam
```

### 8. Key Definitions

- **Classification fine-tuning:** Adapting a pretrained model using labeled examples so that it predicts a discrete class.

- **Class imbalance:** A dataset condition in which some classes contain substantially more examples than others.

- **Undersampling:** Reducing the number of examples from a majority class to create a more balanced dataset.

- **Classification head:** The final layer that maps model representations to class logits.

- **Class logit:** A raw score produced by the model for a possible class before softmax.

- **Binary classification:** A classification problem with exactly two possible target classes.

- **Last-token representation:** The contextual representation at the final sequence position, which in a causal transformer contains information from the preceding tokens.

- **Fine-tuning:** Updating pretrained model parameters using task-specific training data.

- **Frozen parameter:** A model parameter whose gradient updates are disabled during fine-tuning.

### 9. Q/As
- **Q: The dataset is divided into three parts: training, validation, and testing. What is the purpose of each?**   
The **training set** is used to train the model,   
the **validation set** is used to adjust hyperparameters and prevent overfitting, and   
the **testing set** is used to evaluate the model's performance on unseen data. 

- **Q: Why is it often sufficient to fine-tune only the last layers of a pretrained LLM for a new task?**   
**The lower layers** of a pretrained LLM typically capture general language structures and semantics, while **the upper layers** learn task-specific features.  
**Fine-tuning only the last layers (i.e. the upper layers)** allows for efficient adaptation to new tasks without disrupting the learned general language knowledge.

- **Q: Why is the last token in a sequence considered the most informative for classification tasks using a causal attention mask?**  
The causal attention mask restricts each token's attention to itself and preceding tokens.  
As a result, the last token accumulates information from all previous tokens, making it the most comprehensive representation of the input sequence. 

- **Q: What factors influence the choice of the number of epochs during fine tuning?**  
The number of epochs depends on the **dataset's complexity and the task's difficulty**. **Overfitting** may necessitate **reducing the number of epochs**, while **insufficient training** might require **increasing them**.
